## 🔧 Environment Setup

In [ ]:
!nvidia-smi

## Installing ultralytics

In [ ]:
!pip install ultralytics

## 🦏 Training the YOLOv8 Animal Detection Model

We are training a YOLOv8 small model (`yolov8s`) on our African wildlife dataset, which contains four animal classes: **rhino, zebra, elephant, and buffalo**.  

The training command below will:

1. Load the dataset configuration from `african-wildlife.yaml` (contains paths to images and class names).  
2. Start training from the pre-trained `yolov8s.pt` weights.  
3. Run for **30 epochs**, updating the model weights each time it sees all the images.  
4. Resize all images to **640×640 pixels** for consistent input size.  
5. Process **8 images per batch** to balance GPU memory and training speed.  

The model will learn to detect these animals in images and videos, and the best weights will be saved for inference and tracking.

In [ ]:
!yolo detect train data=african-wildlife.yaml model=yolov8s.pt epochs=30 imgsz=640 batch=8

## 📂 Mount Google Drive

- Mount Google Drive to save/load models, images, videos, and outputs.
- Enables persistent storage beyond Colab's temporary environment.

In [ ]:
from ultralytics import YOLO
from google.colab import drive

In [ ]:
#mount google drive
drive.mount('/content/drive')

## 🐾 Load Trained Model

In [ ]:
#load trained animal detection model
model = YOLO("/content/drive/MyDrive/best.pt")

## 🖼️ Run Object Detection on an Image

In [ ]:
#image detection
image_path = "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/images/test/1 (171).jpg"
results = model(image_path, save=True)  # saves output in /content/runs/detect/predict

## 🎥 Track Animals in Videos

- Tracks multiple animals across frames in a video and assigns consistent IDs.
- Saves tracked videos in `/content/runs/track/predict`.
- Handles multiple videos with a loop for batch processing.


In [ ]:
#video tracking
video_list = [
    ("rhino_and_baby", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4"),
    ("elephants", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Elephants.mp4"),
    ("zabras", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Zebras.mp4")
]

#save demo videos
for name, path in video_list:
    model.track(path, save=True)

## 📡 Stream Tracking Results in Memory

- Streams tracking results frame-by-frame for analysis or visualization.
- Allows inspecting boxes, IDs, and classes without saving the video.
- Useful for building datasets for further analysis.

In [ ]:
results = model.track(track_video_path, stream=True)

for r in results:
    print(r.boxes)

## 📊 Build Tracking Dataset from Videos

- Extracts per-frame information for each tracked animal:  
  - `frame number`, `animal ID`, `class name`, `confidence`, `bounding box coordinates`.
- Computes additional features: center coordinates, movement, behavior, and welfare alerts.
- Generates a DataFrame for analysis and Power BI visualization.

In [ ]:
import pandas as pd
import numpy as np

# list of videos (same as saved demos)
video_list = [
    ("rhino_and_baby", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Rhino and Baby.mp4"),
    ("elephants", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Elephants.mp4"),
    ("zabras", "/content/drive/MyDrive/Colab Notebooks/Animal Detection/african-wildlife/videos/test/Zebras.mp4")
]

rows = []

# track and build dataset
for video_name, track_video_path in video_list:
    print(f"Processing video: {video_name}")

    # track video in memory
    results = model.track(track_video_path, stream=True)

    for frame_idx, r in enumerate(results):
        if r.boxes is None or r.boxes.id is None:
            continue

        for i in range(len(r.boxes)):
            x1, y1, x2, y2 = r.boxes.xyxy[i].tolist()
            conf = float(r.boxes.conf[i])
            cls = int(r.boxes.cls[i])
            track_id = int(r.boxes.id[i])
            class_name = model.names[cls]

            rows.append([
                video_name,
                frame_idx,
                track_id,
                class_name,
                conf,
                x1, y1, x2, y2
            ])

# create dataframe
df = pd.DataFrame(rows, columns=[
    "video", "frame", "animal_id", "class_name", "confidence", "x1", "y1", "x2", "y2"
])

## 🚶 Calculate Animal Movement and Classify Behavior

- Computes the center of bounding boxes.  
- Calculates frame-to-frame movement (Euclidean distance).  
- Classifies animal behavior based on movement: `resting`, `walking`, `running`.  
- Adds a welfare alert if movement is low.

In [ ]:
# Calculate center of bounding box
df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

In [ ]:
# Sort data
df = df.sort_values(["animal_id","frame"])

# Previous positions
df["prev_x"] = df.groupby("animal_id")["center_x"].shift(1)
df["prev_y"] = df.groupby("animal_id")["center_y"].shift(1)

# Movement calculation
df["movement"] = np.sqrt(
    (df["center_x"] - df["prev_x"])**2 +
    (df["center_y"] - df["prev_y"])**2
)

df["movement"] = df["movement"].fillna(0)

In [ ]:
def classify_behaviour(m):

    if m < 2:
        return "resting"
    elif m < 20:
        return "walking"
    else:
        return "running"

df["behaviour"] = df["movement"].apply(classify_behaviour)

In [ ]:
def welfare_alert(row):

    if row["movement"] < 10:
        return "Low Activity"

    return "Normal"

df["welfare_alert"] = df.apply(welfare_alert, axis=1)

## 💾 Save Tracking Data for Analysis

- Exports processed tracking data to a CSV file.  
- CSV can be imported into Power BI, Excel, or SQL for reporting and dashboards.


In [ ]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/Animal Detection/animal_tracking.csv", index=False)